# 04 — Spatial Binning (Full Scale)

Verifies spatial coverage overlap between the full plant existence matrix F
and pollinator existence matrix P at 0.5° resolution across CONUS.

The F and P matrices are built in notebooks 01 and 03 respectively.
This notebook checks bin alignment, computes the shared bin set (3,162 CONUS bins),
and confirms that F and P columns are aligned before downstream computation of N.

**Critical:** N (shared bin count per pair) must be computed as
`F_common @ P_common` where both matrices are restricted to the same 3,162
common bins. Never use misaligned F and P arrays.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE    = Path("/scratch/ariana.l")
F_PATH  = BASE / "Stage 4 Link Prediction Model" / "stage4_F_existence_phenofield.csv"
P_PATH  = BASE / "New Stage 4 Link Prediction Model" / "stage4_P_existence_corrected.csv"

print("Paths OK")

In [ ]:
# Load F and P matrices
print("Loading F matrix...")
F_df = pd.read_csv(F_PATH, index_col=0)
print(f"  F shape: {F_df.shape}")

print("Loading P matrix...")
P_df = pd.read_csv(P_PATH, index_col=0)
print(f"  P shape: {P_df.shape}")

In [ ]:
# Verify bin alignment
f_bins = set(F_df.columns)
p_bins = set(P_df.columns)
common_bins = sorted(f_bins & p_bins)

print(f"F bins:      {len(f_bins):,}")
print(f"P bins:      {len(p_bins):,}")
print(f"Common bins: {len(common_bins):,}")
print(f"F-only bins: {len(f_bins - p_bins):,}")
print(f"P-only bins: {len(p_bins - f_bins):,}")

# Confirm bin format
print(f"\nSample bins: {common_bins[:3]}")
# Expected format: '24.5_-81.0'

In [ ]:
# Restrict both matrices to common bins
F_common = F_df[common_bins]
P_common = P_df[common_bins]

print(f"F_common shape: {F_common.shape}")
print(f"P_common shape: {P_common.shape}")

# Spot-check N for a sample pair
sample_plant = F_common.index[0]
sample_pollinator = P_common.index[0]

N_sample = int(F_common.loc[sample_plant].values @ P_common.loc[sample_pollinator].values)
print(f"\nSample N ({sample_plant} × {sample_pollinator}): {N_sample} shared bins")

In [ ]:
# Summary statistics
print("F matrix sparsity:", f"{1 - F_common.values.mean():.4f}")
print("P matrix sparsity:", f"{1 - P_common.values.mean():.4f}")

print(f"\nBin coverage summary:")
print(f"  Bins where ≥1 plant species present:     {(F_common.sum(axis=0) > 0).sum():,}")
print(f"  Bins where ≥1 pollinator species present: {(P_common.sum(axis=0) > 0).sum():,}")
print(f"  Both present:                             {((F_common.sum(axis=0) > 0) & (P_common.sum(axis=0) > 0)).sum():,}")